# Lesson 08 Lab — The OpenAI-Compatible HTTP Service

**Puzzle:** Does API compatibility mean every endpoint and field behaves identically?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A compatible endpoint lowers client migration cost, but it does not erase model capabilities, server-specific fields, parser requirements, or release differences. The contract must be tested against the exact server build and model.


## 0. Predict before running

1. Predict which endpoint will identify the served model.
2. List required Chat response fields.
3. Name one OpenAI feature that must be probed rather than assumed.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab launches `vllm serve` as a child process, waits for readiness, calls `/v1/models` and `/v1/chat/completions`, captures status and timing, then terminates the server cleanly. Unsupported probes remain explicit.

- HTTP 200 does not imply semantic correctness.
- Endpoint availability depends on model task and server configuration.
- Server startup, request latency, and engine execution need separate evidence.


## 2. Derive the mechanism

The server translates an HTTP request into tokenizer, scheduler, sampling, and streaming operations. Compatibility is endpoint- and field-level: a model may support Chat but not embeddings, a tool parser may require flags, and extra vLLM parameters can extend the schema. Readiness, request success, and response structure are separate checks.

### Mechanism at a glance

```mermaid
sequenceDiagram
  participant C as Client
  participant A as API server
  participant E as vLLM engine
  C->>A: GET /v1/models
  A-->>C: served model identity
  C->>A: POST /v1/chat/completions
  A->>E: tokenize + schedule
  E-->>A: generated token stream
  A-->>C: compatible JSON response
```

### Walk it step by step

1. **Wait for readiness.** Do not mix server startup time with request failure.
2. **Probe model identity.** Confirm which name clients must send.
3. **Validate required fields.** Check choices, message content, finish reason, and usage.
4. **Test the production path.** Repeat through authentication, TLS, gateway, and streaming layers.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 8
LESSON_TITLE = 'The OpenAI-Compatible HTTP Service'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260820
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | offline generation only |
| Candidate | localhost OpenAI-compatible HTTP serving |
| Held constant | model, port, sampling, prompt, timeout, and server arguments |
| Measurements | startup time, status codes, response schema, token usage, request latency, and shutdown |
| Evidence | `native-backend` |

**Experiment:** Start the real server on localhost, issue model and Chat requests, validate their JSON shape, and preserve a log tail.


## 5. Inspect the experiment code

The subprocess receives an argument list rather than a shell command. The code polls readiness with a deadline, records a bounded log tail, and always terminates the process in a `finally` block.

Do not execute until the code matches the frozen table.


In [2]:
payload={"model":str(MODEL),"messages":[{"role":"user","content":"Reply with four words about KV cache."}],
         "temperature":0.0,"max_tokens":16,"seed":SEED}
probe=run_server_probe(18018,request_payload=payload); chat=probe["chat_json"]
choice=(chat.get("choices") or [{}])[0]; usage=chat.get("usage") or {}
schema_valid=(isinstance(chat.get("id"),str) and isinstance(chat.get("choices"),list)
              and isinstance(choice.get("message",{}).get("content"),str)
              and "finish_reason" in choice and isinstance(usage,dict))
metrics={"server_ready":probe["server_ready"],"startup_s":probe["startup_s"],
         "models_status":probe["models_status"],"chat_status":probe["chat_status"],
         "chat_latency_s":probe["chat_latency_s"],"completion_tokens":int(usage.get("completion_tokens",0)),
         "schema_valid":schema_valid,"model_ids":[Path(x.get("id","")).name for x in probe["model_json"].get("data",[])],
         "response_preview":choice.get("message",{}).get("content","")[:160],"server_log_tail":SERVER_LOG_TAIL}
analysis=(f"The server became ready in {metrics['startup_s']:.2f} s; models/chat returned "
          f"HTTP {metrics['models_status']}/{metrics['chat_status']} and schema valid={schema_valid}. "
          "This covers one non-streaming localhost route.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Server ready | yes |
| Startup | 20.045628 |
| Models status | 200 |
| Chat status | 200 |
| Chat latency | 0.100528 |
| Completion tokens | 7 |
| Schema valid | yes |


## 7. Explain the result

The server became ready in 20.05 s; models/chat returned HTTP 200/200 and schema valid=True. This covers one non-streaming localhost route.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 8, "title": 'The OpenAI-Compatible HTTP Service', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The localhost test proves the selected Chat route and response schema for this model/server pair; compatibility beyond that matrix remains unmeasured.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 8,
  "title": "The OpenAI-Compatible HTTP Service",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260820
  },
  "evidence_label": "native-backend",
  "metrics": {
    "server_ready": true,
    "startup_s": 20.045627764891833,
    "models_status": 200,
    "chat_status": 200,
    "chat_latency_s": 0.10052789002656937,
    "completion_tokens": 7,
    "schema_valid": true,
    "model_ids": [
      "Qwen2.5-1.5B-Instruct"
    ],
    "response_preview": "Fast access, high performance.",
    "server_log_tail": " 08-13 00:18:30 [launcher.py:46] Route: /v1/responses/{response_id}/cancel, Methods: POST\n(APIServer pid=650324) INFO 08-13 00:18:30 [launcher.py:46] Route: /v1/completions, Methods: POST\n(APIServer pid=650324) INFO 08-13 00:18:30 [launcher.py:46] Route: /v1/messages,

## 9. Make the bounded decision

> The localhost test proves the selected Chat route and response schema for this model/server pair; compatibility beyond that matrix remains unmeasured.

**Acceptance/rollback:** Enable a client route only when required endpoints, fields, streaming behavior, errors, and authentication controls pass contract tests.

**Failure analysis:** Loopback tests exclude proxies, TLS, network jitter, load balancing, and multi-tenant controls. A single response cannot validate all compatibility or parser combinations.


## 10. Extend the evidence

Run a versioned contract suite for Chat, Responses, embeddings, streaming, errors, cancellation, tools, and usage accounting through the production gateway.

The full boundary and references are in [`README.md`](README.md).
